In [ ]:
import os
import json
import gc
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torchcrf import CRF
from IPython.display import display

# ==========================================
# 0. CONFIGURATION - CHỈ CHỈNH SỬA Ở ĐÂY
# ==========================================
CHOSEN_MODEL = "roberta-base"
DATASET_FILE = "/content/NLP/data/dataset.jsonl"

# Gõ "ALL" để chạy đủ 6 cấu trúc
# Hoặc 1 trong 6 tên sau:
# "Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss", "CRF", "CRF+FocalLoss"
RUN_MODE = "ALL"

# NÊN lưu ở /content trước để tránh đầy Google Drive
# Sau khi chạy xong có thể zip/tải về hoặc copy thủ công
SAVE_ROOT = "/content/roberta_results"
os.makedirs(SAVE_ROOT, exist_ok=True)

print(f"🌟 CHUẨN BỊ TRAINING CHO MODEL: {CHOSEN_MODEL.upper()} 🌟")
print(f"🎯 CHẾ ĐỘ CHẠY: {RUN_MODE}")
print(f"💾 THƯ MỤC LƯU: {SAVE_ROOT}")
print("=" * 70)

# ==========================================
# 1. LOAD & PREPARE DATA
# ==========================================
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

all_tags = [tag for tags_list in dataset["train"]["ner_tags"] for tag in tags_list]
unique_tags = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(unique_tags)}
id2label = {i: label for i, label in enumerate(unique_tags)}

def encode_tags(example):
    example["ner_tags_id"] = [label2id.get(tag, 0) for tag in example["ner_tags"]]
    return example

dataset = dataset.map(encode_tags)

# Tính toán Class Weights dùng cho Focal Loss
tag_counts = pd.Series(all_tags).value_counts()
total_tags = len(all_tags)
class_weights = [total_tags / (len(unique_tags) * tag_counts[label]) for label in unique_tags]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# ==========================================
# 2. DEFINING CUSTOM LOSSES & TRAINERS
# ==========================================
# --- FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss_raw = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss_raw)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss_raw
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_loss = focal_loss * alpha
        return focal_loss.mean()

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        active_loss = inputs["attention_mask"].view(-1) == 1
        active_logits = logits.view(-1, model.config.num_labels)
        active_labels = torch.where(
            active_loss,
            labels.view(-1),
            torch.tensor(-100, device=labels.device)
        )

        valid_indices = active_labels != -100
        valid_logits = active_logits[valid_indices]
        valid_labels = active_labels[valid_indices]

        focal_loss_fn = FocalLoss(weight=class_weights_tensor.to(model.device), gamma=2.0)
        loss = focal_loss_fn(valid_logits, valid_labels)
        return (loss, outputs) if return_outputs else loss

# --- DICE LOSS ---
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, ignore_index=-100):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.shape[-1]
        probs = F.softmax(logits, dim=-1).view(-1, num_classes)
        targets = targets.view(-1)

        valid_mask = targets != self.ignore_index
        valid_targets = targets[valid_mask]
        valid_probs = probs[valid_mask]

        if valid_targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        targets_one_hot = F.one_hot(valid_targets, num_classes=num_classes).float()
        intersection = torch.sum(valid_probs * targets_one_hot, dim=0)
        cardinality = torch.sum(valid_probs + targets_one_hot, dim=0)

        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1.0 - dice_score).mean()

class DiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        dice_loss_fn = DiceLoss(ignore_index=-100)
        loss = dice_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- CE + DICE LOSS TRAINER ---
class CEDiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        ce_loss = outputs.loss
        logits = outputs.logits

        dice_loss_fn = DiceLoss(ignore_index=-100)
        dice_loss = dice_loss_fn(logits, labels)

        total_loss = ce_loss + dice_loss
        return (total_loss, outputs) if return_outputs else total_loss

# ==========================================
# 3. CRF ARCHITECTURE (WITH OPTIONAL FOCAL)
# ==========================================
class TransformerCRF(nn.Module):
    def __init__(self, model_name, num_labels, use_focal_loss=False, gamma=2.0):
        super(TransformerCRF, self).__init__()
        self.num_labels = num_labels
        self.use_focal_loss = use_focal_loss
        self.gamma = gamma
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state)

        loss = None
        if labels is not None:
            mask = attention_mask.bool()
            safe_labels = torch.where(labels == -100, torch.tensor(0, device=labels.device), labels)

            if self.use_focal_loss:
                nll = -self.crf(logits, tags=safe_labels, mask=mask, reduction='none')
                pt = torch.exp(-nll)
                loss = (((1 - pt) ** self.gamma) * nll).mean()
            else:
                loss = -self.crf(logits, tags=safe_labels, mask=mask, reduction='mean')

        eval_mask = attention_mask.bool()
        decoded_tags = self.crf.decode(logits, mask=eval_mask)

        fake_logits = torch.zeros_like(logits)
        for i, tags in enumerate(decoded_tags):
            for j, tag in enumerate(tags):
                fake_logits[i, j, tag] = 1.0

        return {"loss": loss, "logits": fake_logits} if loss is not None else {"logits": fake_logits}

# ==========================================
# 4. EVALUATION METRICS
# ==========================================
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[pred] for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[lab] for (pred, lab) in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ==========================================
# 5. EXPORT FUNCTION (THỐNG NHẤT CHO TẤT CẢ)
# ==========================================
def export_model(model, tokenizer, arch_type, save_dir, id2label, label2id):
    os.makedirs(save_dir, exist_ok=True)
    print(f"\n💾 Đang tiến hành lưu model ({arch_type}) tại: {save_dir} ...")

    # 1. Lưu tokenizer
    tokenizer.save_pretrained(save_dir)

    # Một số tokenizer support save_vocabulary, một số sẽ lỗi -> bọc try/except
    try:
        if hasattr(tokenizer, "save_vocabulary"):
            tokenizer.save_vocabulary(save_dir)
    except Exception as e:
        print(f"⚠️ Bỏ qua save_vocabulary(): {e}")

    # 2. Lưu config + label mapping
    if hasattr(model, "config"):
        model.config.id2label = id2label
        model.config.label2id = label2id
        model.config.save_pretrained(save_dir)

    # 3. Lưu trọng số
    if arch_type in ["CRF", "CRF+FocalLoss"]:
        torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))

        with open(os.path.join(save_dir, "labels.json"), "w", encoding="utf-8") as f:
            json.dump(
                {"id2label": id2label, "label2id": label2id},
                f,
                ensure_ascii=False,
                indent=4
            )

        print("✅ Đã lưu: Tokenizer + CRF Weights (.bin) + Config + Labels")
    else:
        model.save_pretrained(save_dir, safe_serialization=True)
        print("✅ Đã lưu: Tokenizer + HF Model (.safetensors) + Config")

    files = os.listdir(save_dir)
    print(f"📦 Các file hiện có trong folder: {files}")

# ==========================================
# 6. CORE TRAINING ENGINE
# ==========================================
def run_experiment(model_name, arch_type):
    base_name = model_name.split('-')[0].upper()
    arch_name = f"{base_name} + {arch_type}"
    print(f"\n🚀 Đang chạy: {arch_name}...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        add_prefix_space=True if "roberta" in model_name else False
    )

    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(
            examples["tokens"],
            truncation=True,
            is_split_into_words=True
        )
        labels = []
        for i, label in enumerate(examples["ner_tags_id"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            labels.append(label_ids)
        tokenized_inputs["labels"] = labels
        return tokenized_inputs

    tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    # Chọn model + trainer class
    if arch_type == "Base_CE":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(unique_tags),
            id2label=id2label,
            label2id=label2id
        )
        trainer_class = Trainer

    elif arch_type == "FocalLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(unique_tags),
            id2label=id2label,
            label2id=label2id
        )
        trainer_class = FocalLossTrainer

    elif arch_type == "DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(unique_tags),
            id2label=id2label,
            label2id=label2id
        )
        trainer_class = DiceLossTrainer

    elif arch_type == "CE+DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(unique_tags),
            id2label=id2label,
            label2id=label2id
        )
        trainer_class = CEDiceLossTrainer

    elif arch_type == "CRF":
        model = TransformerCRF(
            model_name,
            num_labels=len(unique_tags),
            use_focal_loss=False
        )
        trainer_class = Trainer

    elif arch_type == "CRF+FocalLoss":
        model = TransformerCRF(
            model_name,
            num_labels=len(unique_tags),
            use_focal_loss=True,
            gamma=2.0
        )
        trainer_class = Trainer

    else:
        raise ValueError(f"Unsupported architecture: {arch_type}")

    # KHÔNG save checkpoint giữa chừng để đỡ đầy bộ nhớ
    safe_arch_type = arch_type.replace("+", "_").replace(" ", "_")
    training_args = TrainingArguments(
        output_dir=os.path.join(SAVE_ROOT, f"temp_{safe_arch_type}"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none"
    )

    trainer = trainer_class(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    f1_score = eval_results["eval_f1"]

    final_save_dir = os.path.join(SAVE_ROOT, f"final_{safe_arch_type}")
    os.makedirs(final_save_dir, exist_ok=True)

    with open(os.path.join(final_save_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(eval_results, f, ensure_ascii=False, indent=4)

    export_model(
        model=model,
        tokenizer=tokenizer,
        arch_type=arch_type,
        save_dir=final_save_dir,
        id2label=id2label,
        label2id=label2id
    )

    print(f"✅ Xong {arch_name}! F1: {f1_score:.4f}")
    print(f"💾 Model đã lưu tại: {final_save_dir}")

    result = {
        "Architecture": arch_type,
        "F1-Score": f1_score,
        "Precision": eval_results["eval_precision"],
        "Recall": eval_results["eval_recall"],
        "Accuracy": eval_results["eval_accuracy"]
    }

    # Dọn RAM/GPU sau mỗi architecture
    del trainer
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

# ==========================================
# 7. EXECUTION & DISPLAY
# ==========================================
VALID_ARCHS = [
    "Base_CE",
    "FocalLoss",
    "DiceLoss",
    "CE+DiceLoss",
    "CRF",
    "CRF+FocalLoss"
]

if RUN_MODE == "ALL":
    architectures = VALID_ARCHS
elif RUN_MODE in VALID_ARCHS:
    architectures = [RUN_MODE]
else:
    raise ValueError(f"❌ RUN_MODE không hợp lệ! Vui lòng chọn 'ALL' hoặc một trong: {VALID_ARCHS}")

results_list = []

for arch in architectures:
    try:
        res = run_experiment(CHOSEN_MODEL, arch_type=arch)
        results_list.append(res)
    except Exception as e:
        print(f"❌ Lỗi khi chạy {CHOSEN_MODEL} + {arch}: {e}")

print("\n" + "*" * 70)
print(f"🏆 BÁO CÁO KẾT QUẢ CHO: {CHOSEN_MODEL.upper()} 🏆")
print("*" * 70)

df_results = pd.DataFrame(results_list)

if not df_results.empty:
    df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
    display(df_results)

    best_arch = df_results.iloc[0]["Architecture"]
    best_f1 = df_results.iloc[0]["F1-Score"]

    if len(architectures) > 1:
        print(f"\n🎉 Cấu trúc ĐỈNH NHẤT cho model này là: {best_arch} (F1: {best_f1:.4f}) 🎉")
    else:
        print(f"\n✅ Hoàn thành test độc lập cấu trúc: {best_arch} (F1: {best_f1:.4f})")
else:
    print("❌ Không có kết quả nào được tạo ra.")